### Dependencies

In [1]:
from apify_client import ApifyClient
from dotenv import load_dotenv
import pandas as pd
import re
import requests
import json
import os
import shutil
from datetime import datetime
from pathlib import Path
from apify_class import Apify

In [157]:
import importlib
import apify_class

importlib.reload(apify_class)

<module 'apify_class' from 'c:\\Users\\ismai\\OneDrive\\Desktop\\UpClout\\src\\apify_class.py'>

In [3]:
DATA_PATH = "../data"

In [2]:
def clean_post_data(current_folder="../data/sajalaly") -> list[dict]:

    json_file = None
    for file in os.listdir(current_folder):
        if file.endswith(".json"):
            json_file = os.path.join(current_folder, file)
            break

    with open(json_file, "r", encoding="utf-8") as file:
        data=json.load(file)
   
    return data

dict_list = clean_post_data()

In [ ]:
_dict = dict_list[0]

for k in _dict.keys():
    print(k)

In [ ]:
_dict

In [ ]:
keys_of_interest = [
    "id",
    "type",
    "caption",
    "hashtags", #list
    "mentions", #list
    "url",
    "commentsCount",
    "displayUrl",
    "images", #list
    "likesCount",
    "videoViewCount",
    "timestamp",
    "ownerUsername",
    "ownerId",
    "taggedUsers",
    "coauthorProducers",
    "videoPlayCount",
    "ownerFullName",
    "videoDuration",
    "isSponsored",
]

In [10]:
new_dict = {k: v for k, v in _dict.items() if k in keys_of_interest}

In [11]:
new_dict

{'id': '3735340024347818050',
 'type': 'Video',
 'caption': 'Breakups hurt… except this one. 😉\n\nDitch your old phone and join Sajal in the Ultra league.\n\nSuper 26 Ultra is here to redefine your every day. Don’t just move on, move up. Get yours now!\n\n#itel #itelPakistan #UltraSmooth #UltraStrong #Super26Ultra',
 'hashtags': ['itel',
  'itelPakistan',
  'UltraSmooth',
  'UltraStrong',
  'Super26Ultra'],
 'mentions': [],
 'url': 'https://www.instagram.com/p/DPWl_YijPhC/',
 'commentsCount': 143,
 'displayUrl': 'https://scontent-sin2-1.cdninstagram.com/v/t51.2885-15/559551741_18090698341862474_6134346958603881048_n.jpg?stp=dst-jpg_e15_fr_p1080x1080_tt6&_nc_ht=scontent-sin2-1.cdninstagram.com&_nc_cat=102&_nc_oc=Q6cZ2QGuT2wBlnR5rhcEnMSpjk3Dsh1YUgz6zt_Kd1xLTfaiSok-WFARayjaewtyd0dkpNs&_nc_ohc=PoAfBwD392wQ7kNvwENknOT&_nc_gid=vCr_EcCWgajuFZyIrG9gGQ&edm=APs17CUBAAAA&ccb=7-5&oh=00_Afc0HrKr8NdqhrglNhEXCmU17HY5pGHwuTFKEg0n6TIr1w&oe=68E87458&_nc_sid=10d13b',
 'images': [],
 'likesCount': 6749,
 

In [ ]:
def handle_hashtags(postID: int, hashtags: list) -> None:
    if not hashtags:
        return
    
    # Dump Hashtags in Hashtags table first
    for hashtag in hashtags:
        # load in hashtags table
        # load.load_hashtags_table(hashtag)
        ...
    # Dumping in Hastags_Posts (N:M) table
    # load.load_posts_hashtags_table(postID, hashtag)

    

In [155]:
def potential_influencers(username: str):
    # scrape meta deta of username
    # check if following is greater than threshold (1000) 
    # append username to txt file
    # else delete scraped data
    apify = Apify()
    response=apify.scrape_meta_data(username)

    print(response)

    if response == 0:
        print("Already in Database")
        return

    folder_path = f"../data/{username}"
    df=pd.read_csv(f"{folder_path}/{username}_meta_data.csv")

    threshold: int = 1000
    followers = int(df['followersCount'][0])
    if followers > threshold:
        print("Potential Influencer")
    else:
        # delete folder
        if os.path.exists(folder_path):
            shutil.rmtree(folder_path)
            print(f"Folder {folder_path} deleted successfully!")

In [ ]:
def check_mentions(new_dict: dict) -> None:
    mentions = new_dict['mentions']
    if not mentions:
        return
    
    for username in mentions:
        # TODO: send username to a function that determines if it is indeed an influencer worth keeping in database 
        #potential_influencers(username)
        ...

In [162]:
potential_influencers("humzaamin")

Folder for humzaamin already exists. Skipping...
0
Already in Database


In [ ]:
check_mentions(new_dict)

In [163]:
new_dict['mentions']

['emaandharani',
 'emaandharanistyled',
 'iambabarzaheer',
 'mubsher.bhatti',
 'shahbazshaziofficial']

In [164]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

url1="https://starngage.com/plus/en/brand/ranking/instagram/pakistan/politics"

url="https://starngage.com/plus/en/influencer/ranking/instagram/pakistan"
driver = webdriver.Chrome()  # or webdriver.Firefox()
driver.get(url)

# Wait for the table to load
wait = WebDriverWait(driver, 10)
tbody = wait.until(EC.presence_of_element_located((By.TAG_NAME, "tbody")))

# Find all name links
name_links = driver.find_elements(By.CSS_SELECTOR, "tbody tr .name a")
names = [link.text for link in name_links if link.text.strip()]

driver.quit()

cleaned_names = [name.lstrip('@') for name in names]

print(len(cleaned_names))

with open("../insta_profiles.txt", "a") as file:
    for username in cleaned_names:
        file.write(username + "\n")

100


In [140]:
with open("../data/brand_data.json", "r", encoding="utf-8") as file:
    data=json.load(file)

top_posts=data[1]['topPosts']

In [141]:
top_posts[29]['mentions']

[]

In [142]:
usernames=[]
mentions=[]

for index_2 in range(29):
    try:
        mentions.append(top_posts[index_2]['mentions'])
    except IndexError as e:
        print(f"Stopped at inner length: {index_2}\nError: {e}")

In [107]:
usernames=set(usernames)

In [143]:
mentions=[lst for lst in mentions if lst]
# Flatten the list
mentions = [username for sublist in mentions for username in sublist]

In [129]:
len(usernames)

0

In [144]:
len(mentions)

7

In [145]:
mentions

['nishatemporium',
 'Winter',
 'oriflamewithnazish',
 'am_brandstore',
 '03043888895',
 'khizan_official1',
 'khizanbts']

In [104]:
with open("../brand_profiles.txt", "a") as file:
    for username in usernames:
        file.write(username + "\n")
